# Jack The Walker - Full Training Pipeline

Train Jack to walk, understand physics, and be a companion.

**Phases:**
- Phase 0: Physics Foundation (SymPy supervision on MuJoCo rollouts)
- Phase 2: Locomotion RL (PPO with RL-Zoo3 Humanoid-v4 tuned hyperparams)
- Phase 8: Companion (Emotional dynamics + movement-mood coupling)

**Requirements:** Colab with GPU runtime (T4 or better)

**Time estimates:**
- Phase 0: ~15-30 minutes
- Phase 2: ~2-8 hours (2M timesteps)
- Phase 8: ~30-60 minutes

## 1. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/YOUR_USERNAME/JackTheWalker.git
%cd JackTheWalker

In [ ]:
# Install dependencies
!pip install torch torchvision --quiet
!pip install mujoco gymnasium[mujoco] --quiet
!pip install sympy tqdm numpy --quiet
!pip install transformers --quiet  # For LLM (optional, Phase 3+)

In [ ]:
# Verify GPU and MuJoCo
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

import gymnasium as gym
env = gym.make('Humanoid-v5')
print(f'Humanoid-v5: obs={env.observation_space.shape}, act={env.action_space.shape}')
env.close()
print('\nAll good! Ready to train.')

## 2. (Optional) Mount Google Drive for checkpoint persistence

In [ ]:
# Mount Drive so checkpoints survive Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/JackTheLearner/checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive checkpoint dir: {DRIVE_DIR}')

## 3. Phase 0: Physics Foundation

Teaches Jack basic physics from MuJoCo rollouts with SymPy ground truth.
Fills the replay buffer and computes EWC Fisher information.

**~15-30 minutes on T4 GPU**

In [ ]:
from TrainingPipeline import TrainingPipeline, PipelineConfig

config = PipelineConfig(
    checkpoint_dir='checkpoints',
    drive_path=DRIVE_DIR if 'DRIVE_DIR' in dir() else '',
)

pipeline = TrainingPipeline(config)
pipeline.train_phase0(epochs=30, samples_per_epoch=10000)

In [ ]:
# Backup to Drive
if 'DRIVE_DIR' in dir():
    import shutil
    for f in ['phase0_best.pt', 'phase0_latest.pt', 'replay.pt', 'ewc.pt']:
        src = f'checkpoints/{f}'
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_DIR, f))
            print(f'  Backed up {f}')

## 4. Phase 2: Locomotion RL (PPO)

Teaches Jack to WALK using PPO with RL-Zoo3 Humanoid-v4 tuned hyperparameters.

Key hyperparams (from RL-Zoo3):
- lr = 3.57e-5 (10x lower than default)
- gamma = 0.95 (shorter horizon)
- clip_range = 0.3 (wider than default)
- batch_size = 64, n_steps = 512

**~2-8 hours on T4 GPU for 2M timesteps**

You can stop early and resume - checkpoints are saved every 5120 steps.

In [ ]:
# Resume pipeline (loads Phase 0 checkpoint)
from TrainingPipeline import TrainingPipeline, PipelineConfig

config = PipelineConfig(
    checkpoint_dir='checkpoints',
    drive_path=DRIVE_DIR if 'DRIVE_DIR' in dir() else '',
)

pipeline = TrainingPipeline(config)

# Train walking! Adjust timesteps based on available time:
# 500K  = ~30 min (basic standing/stumbling)
# 2M    = ~2-4 hours (walking)
# 5M    = ~5-10 hours (stable walking)
# 10M   = ~10-20 hours (robust walking)
pipeline.train_phase2(total_timesteps=2_000_000)

In [ ]:
# Backup to Drive
if 'DRIVE_DIR' in dir():
    import shutil
    for f in ['phase2_best.pt', 'phase2_latest.pt']:
        src = f'checkpoints/{f}'
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_DIR, f))
            print(f'  Backed up {f}')

## 5. Phase 8: Companion Training

Teaches Jack emotional dynamics and movement-mood coupling.
After this, his walking style changes based on his mood.

**~30-60 minutes on T4 GPU**

In [ ]:
from TrainingPipeline import TrainingPipeline, PipelineConfig

config = PipelineConfig(
    checkpoint_dir='checkpoints',
    drive_path=DRIVE_DIR if 'DRIVE_DIR' in dir() else '',
)

pipeline = TrainingPipeline(config)
pipeline.train_phase8(epochs=50)

In [ ]:
# Final backup
if 'DRIVE_DIR' in dir():
    import shutil
    for f in os.listdir('checkpoints'):
        if f.endswith('.pt'):
            shutil.copy2(f'checkpoints/{f}', os.path.join(DRIVE_DIR, f))
    print('All checkpoints backed up to Drive!')
    print(f'Download from: {DRIVE_DIR}')

## 6. Test Jack

Quick test to verify the trained model works.

In [ ]:
import torch
import gymnasium as gym
import numpy as np
from TrainingPipeline import TrainingPipeline, PipelineConfig

config = PipelineConfig(checkpoint_dir='checkpoints')
pipeline = TrainingPipeline(config)
pipeline.make_optimizer(2)
pipeline.load('phase2_best')

# Run a test episode
env = gym.make('Humanoid-v5')
obs, _ = env.reset()
total_reward = 0
steps = 0

for _ in range(1000):
    obs_norm = pipeline.normalize_obs(obs)
    obs_t = torch.tensor(obs_norm, dtype=torch.float32, device=pipeline.device).unsqueeze(0)
    state = pipeline.project_obs(obs_t)
    
    with torch.no_grad():
        output = pipeline.model(state)
        action = output['actions'][:, 0, :].cpu().numpy()[0]
    
    obs, reward, term, trunc, info = env.step(action)
    total_reward += reward
    steps += 1
    if term or trunc:
        break

env.close()
print(f'Test episode: {steps} steps, reward={total_reward:.1f}')
if steps > 100:
    print('Jack is walking!')
else:
    print('Jack needs more training (increase timesteps in Phase 2)')

## 7. Next Steps

Download the checkpoints and run locally:

```bash
# Copy checkpoints from Drive to your local JackTheWalker/checkpoints/

# Run Jack's virtual world (needs pygame + mujoco locally)
python VirtualWorld.py

# Or text-only mode (works anywhere)
python VirtualWorld.py --text-only
```

Jack will walk around his room, respond to your commands,
have emotions, think his own thoughts, and remember your conversations.